
# AI Quant Research Lab
## Stage 1: Exploratory Data Analysis

### Objective
Analyze historical US stock market data to understand price movements,
calculate daily returns, and validate data quality before developing
and backtesting quantitative trading strategies.

### Asset
Apple Inc. (AAPL)

### Data source
Yahoo Finance, accessed using the yfinance Python library.

### Period
January 2021 to January 2026.

### Research principle
All analysis must be reproducible, and all trading strategies must be
tested against a benchmark with realistic assumptions.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yfinance as yf

print("All libraries imported successfully!")


In [ ]:

ticker = "AAPL"
start_date = "2021-01-01"
end_date = "2026-01-01"

data = yf.download(
    ticker,
    start=start_date,
    end=end_date,
    interval="1d",
    auto_adjust=False,
    progress=False
)

if data.empty:
    raise ValueError(
        "No stock data downloaded. Check your internet connection "
        "and try again."
    )

print("Data downloaded successfully!")
print("Number of rows:", len(data))
print("Number of columns:", len(data.columns))

display(data.head())


In [ ]:

# Standardize columns for a single downloaded ticker
if isinstance(data.columns, pd.MultiIndex):
    data.columns = data.columns.get_level_values(0)

# Remove any accidental duplicate column labels
data = data.loc[:, ~data.columns.duplicated()].copy()

# Sort observations from oldest to newest
data = data.sort_index()

print("Standardized columns:")
print(data.columns.tolist())

print("\nDate range:")
print(data.index.min(), "to", data.index.max())

display(data.head())


In [ ]:

print("DATA-INTEGRITY AUDIT")
print("=" * 45)

# 1. Dataset overview
print("\n1. DATASET OVERVIEW")
print("Rows:", len(data))
print("Columns:", len(data.columns))
print("Start date:", data.index.min())
print("End date:", data.index.max())

# 2. Missing values
print("\n2. MISSING VALUES")
print(data.isna().sum())

# 3. Duplicate dates and rows
print("\n3. DUPLICATES")
print("Duplicate dates:", data.index.duplicated().sum())
print("Duplicate rows:", data.duplicated().sum())

# 4. Infinite numeric values
print("\n4. INFINITE VALUES")
numeric_data = data.select_dtypes(include="number")
print(
    "Total infinite values:",
    np.isinf(numeric_data.to_numpy(dtype=float)).sum()
)

# 5. Non-positive prices
print("\n5. NON-POSITIVE PRICES")
price_columns = [
    col for col in ["Open", "High", "Low", "Close", "Adj Close"]
    if col in data.columns
]

for col in price_columns:
    invalid_count = ((data[col] <= 0) & data[col].notna()).sum()
    print(f"{col}: {invalid_count}")

# 6. Chronological order
print("\n6. DATE ORDER")
print("Chronological:", data.index.is_monotonic_increasing)

# 7. Basic price consistency
print("\n7. PRICE CONSISTENCY")
if {"High", "Low"}.issubset(data.columns):
    invalid_range = (data["High"] < data["Low"]).sum()
    print("Rows where High < Low:", invalid_range)

print("\nAudit complete.")


In [ ]:

# Calculate daily returns using adjusted closing prices
if "Adj Close" not in data.columns:
    raise ValueError(
        "Adjusted Close is missing. Inspect the downloaded columns."
    )

data["Daily_Return"] = data["Adj Close"].pct_change(fill_method=None)

print("DAILY RETURNS")
display(data[["Adj Close", "Daily_Return"]].head(10))

print("\nRETURN STATISTICS")
display(data["Daily_Return"].describe())

print(
    "\nMissing daily returns:",
    data["Daily_Return"].isna().sum()
)


In [ ]:

plt.figure(figsize=(12, 5))

plt.plot(data.index, data["Adj Close"], label="AAPL Adjusted Close")

plt.title("Apple (AAPL) Adjusted Closing Price")
plt.xlabel("Date")
plt.ylabel("Price (USD)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:

plt.figure(figsize=(12, 5))

plt.plot(
    data.index,
    data["Daily_Return"],
    linewidth=0.7,
    label="Daily Return"
)

plt.axhline(0, linewidth=1)
plt.title("Apple (AAPL) Daily Returns")
plt.xlabel("Date")
plt.ylabel("Daily Return")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
